In [30]:
import uuid
from decimal import Decimal
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
import random
from datetime import datetime, timedelta
from faker import Faker
from pyspark.sql.functions import col, when, date_sub, current_timestamp, datediff, max, min
from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType, DateType, TimestampType, DecimalType

In [31]:
def generate_users(fake: Faker, count: int):
    rows = []
    for i in range(1, count + 1):
        rows.append({
            "user_id": i,
            "email": fake.unique.email(),
            "name": fake.name(),
            "age": fake.random_int(min=10, max=100),
            "gender": random.choice(["M", "F"]),
            "job": fake.job(),
            "address": fake.address(),
            "signup": (datetime.now() - timedelta(days=random.randint(0, 365))).date(),
            "created_at": datetime.now()
        })
    return rows


def generate_orders(user_ids: list[int], count: int):
    statuses = ["CREATED", "PAID", "SHIPPED", "DELIVERED", "CANCELED"]
    rows = []
    for _ in range(count):
        status = random.choice(statuses)
        created_at = datetime.now() - timedelta(days=random.randint(0, 60))
        rows.append({
            "order_no": f"ORD-{uuid.uuid4().hex[:20].upper()}",
            "user_id": int(random.choice(user_ids)),
            "status": status,
            "total_amount": Decimal(random.randint(1_000, 500_000)) / Decimal(100),
            "created_at": created_at,
            "updated_at": created_at
        })
    return rows


user_schema = StructType([
    StructField("user_id", LongType(), False),
    StructField("email", StringType(), False),
    StructField("name", StringType(), False),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("job", StringType(), True),
    StructField("address", StringType(), True),
    StructField("signup", DateType(), True),
    StructField("created_at", TimestampType(), True)
])

order_schema = StructType([
    StructField("order_no", StringType(), False),
    StructField("user_id", LongType(), False),
    StructField("status", StringType(), False),
    StructField("total_amount", DecimalType(14, 2), False),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),
])

In [32]:
spark = SparkSession.builder.appName("S3 Spark") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "1000000") \
    .config("spark.hadoop.fs.s3a.vectored.read.min.seek.size", "4K") \
    .config("spark.hadoop.fs.s3a.vectored.read.max.merged.size", "1M") \
    .config("spark.hadoop.fs.s3a.vectored.active.ranged.reads", "4") \
    .config("spark.hadoop.fs.s3a.experimental.input.fadvise", "random") \
    .getOrCreate()

In [33]:
spark.sparkContext.getConf().get("spark.sql.execution.arrow.maxRecordsPerBatch")

'1000000'

In [34]:
users_ = generate_users(Faker("ko_KR"), 1000)
users = spark.createDataFrame(users_, schema=user_schema)
orders = spark.createDataFrame(generate_orders([user["user_id"] for user in users_], 1000), schema=order_schema)

In [35]:
users.show(1), orders.show(1)
users.select("user_id", "name", "gender").show(1), orders.select("order_no", "status", "total_amount").show(1)

+-------+--------------------+------+---+------+---------------------+--------------------------------+----------+--------------------+
|user_id|               email|  name|age|gender|                  job|                         address|    signup|          created_at|
+-------+--------------------+------+---+------+---------------------+--------------------------------+----------+--------------------+
|      1|seoyeongim@exampl...|엄민재| 36|     F|네트워크시스템 개발자|경상북도 보령시 서초대034거리...|2025-06-10|2026-01-18 21:38:...|
+-------+--------------------+------+---+------+---------------------+--------------------------------+----------+--------------------+
only showing top 1 row

+--------------------+-------+--------+------------+--------------------+--------------------+
|            order_no|user_id|  status|total_amount|          created_at|          updated_at|
+--------------------+-------+--------+------------+--------------------+--------------------+
|ORD-D03B252C9CA94...|    254|CANCEL

(None, None)

In [36]:
users.count(), orders.count()
orders.select("user_id").distinct().count()

630

In [37]:
users.filter(col("age") >= 30).count()
orders.filter(col("status") == "PAID").count()
orders.filter((col("status") == "DELIVERED") & (col("total_amount") >= 100_000)).count()

0

In [38]:
orders.orderBy(col("total_amount").desc()).show(1)
users.orderBy("age").show(1)

+--------------------+-------+-------+------------+--------------------+--------------------+
|            order_no|user_id| status|total_amount|          created_at|          updated_at|
+--------------------+-------+-------+------------+--------------------+--------------------+
|ORD-CD4B2B3F34D84...|    451|CREATED|     4995.14|2025-12-12 21:38:...|2025-12-12 21:38:...|
+--------------------+-------+-------+------------+--------------------+--------------------+
only showing top 1 row

+-------+--------------+------+---+------+-------------------------------+------------------------------+----------+--------------------+
|user_id|         email|  name|age|gender|                            job|                       address|    signup|          created_at|
+-------+--------------+------+---+------+-------------------------------+------------------------------+----------+--------------------+
|    421|si@example.net|김정훈| 10|     M|마술사 및 기타 문화/ 예술 관...|대구광역시 강북구 언주54로 308|2025-11-30|2

In [39]:
orders.withColumn(
    "amount_level",
    when(col("total_amount") >= 300_000, "HIGH")
    .when(col("total_amount") >= 100_000, "MID")
    .otherwise("LOW")) \
    .select("order_no", "total_amount", "amount_level") \
    .show(1)

+--------------------+------------+------------+
|            order_no|total_amount|amount_level|
+--------------------+------------+------------+
|ORD-D03B252C9CA94...|     1867.81|         LOW|
+--------------------+------------+------------+
only showing top 1 row



### 최근 7일 주문만 분석

In [40]:
recent_orders = orders.filter(col("created_at") >= date_sub(current_timestamp(), 7))
recent_orders.groupBy("status").count().show(1)

+-------+-----+
| status|count|
+-------+-----+
|CREATED|   25|
+-------+-----+
only showing top 1 row



### 간단한 조인

In [42]:
u = users.alias("u")
o = orders.alias("o")
users_orders = u.join(o, on="user_id", how="inner").select(col("user_id"), col("name"), col("signup"), col("order_no"), col("o.created_at"))
users_orders.show(1)

+-------+------+----------+--------------------+--------------------+
|user_id|  name|    signup|            order_no|          created_at|
+-------+------+----------+--------------------+--------------------+
|    191|최성호|2025-09-26|ORD-D4CF6AA2BDE84...|2025-12-03 21:38:...|
+-------+------+----------+--------------------+--------------------+
only showing top 1 row



### 사용자 가입 후 첫 주문까지 걸린 시간

In [43]:
first_order = users.groupBy("user_id", "name", "signup").agg(F.min("created_at").alias("first_order_at"))
first_order.withColumn("days_to_first_order", datediff("first_order_at", "signup")).select("user_id", "name", "days_to_first_order").show(10)

+-------+------+-------------------+
|user_id|  name|days_to_first_order|
+-------+------+-------------------+
|    265|김준호|                302|
|    339|노정웅|                233|
|    474|이현지|                  5|
|    292|김주원|                 32|
|    349|김은경|                346|
|    444|허우진|                364|
|    415|강서현|                 79|
|    452|이정수|                  3|
|    261|이은정|                 21|
|    271|고정자|                191|
+-------+------+-------------------+
only showing top 10 rows

